# COVID-19 data by epidemiological wave

This notebook filters the consolidated COVID-19 dataset of Mexico for each of the six epidemiological waves and saves one CSV file per wave in `Datos/`. It also writes a CSV with the epidemiological parameters supplied for each phase, using English column titles.

In [1]:
from pathlib import Path

from codes import *

DATA_DIR = Path.cwd().parent / "Datos"
RAW_DIR = DATA_DIR / "raw_data"
WAVE_DIR = DATA_DIR / "waves"
METADATA_DIR = DATA_DIR / "metadata"
SOURCE_CSV = RAW_DIR / "COVID19MEXICO2020_2021_2022_2023.csv"

# Keep every record in the date range (the SIRD/Kalman helpers filter
# positive cases themselves). Set to True to store positive cases only.
POSITIVE_ONLY = False

## Load the consolidated dataset

In [2]:
df = pd.read_csv(SOURCE_CSV)
df = dates_to_datetime(df, ["FECHA_INGRESO", "FECHA_SINTOMAS"])
df.shape

/Users/marcosbautista/Documents/GitHub/Estimation-of-epidemiological-parameters/Codigos/codes.py:12: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['FECHA_DEF'] = pd.to_datetime(df['FECHA_DEF'], errors='coerce')


(20372904, 28)

## Wave definitions

The date ranges follow the monthly intervals of the reference table. The first wave starts on 2020-03-23 so that it does not overlap the pre-lockdown phase (Feb 2020 – Mar 2020), which is already stored in `pre_lockdown_filtered_data_2020-02-28_2020-03-22.csv`.

In [3]:
WAVES = [
    {
        "phase": "Pre-lockdown",
        "interval": "Feb 2020 - Mar 2020",
        "start": "2020-02-28", "end": "2020-03-22",
        "variant": "Ancestral (Wuhan-Hu-1)",
        "beta": "0.5534 – 1.1400",
        "gamma": "0.0708 – 0.2000",
        "delta": "0.0561 – 0.1200",
        "r0": "1.762 – 2.860",
        "csv": "pre_lockdown_filtered_data_2020-02-28_2020-03-22.csv",
        "already_saved": True,
    },
    {
        "phase": "First Wave",
        "interval": "Mar 2020 - Sep 2020",
        "start": "2020-03-23", "end": "2020-09-30",
        "variant": "Ancestral lineages B.1",
        "beta": "0.5534 → 0.1069",
        "gamma": "0.0708 → 0.1383",
        "delta": "0.0561 → 0.0211",
        "r0": "1.100 - 1.300",
    },
    {
        "phase": "Second Wave",
        "interval": "Sep 2020 - Apr 2021",
        "start": "2020-10-01", "end": "2021-04-30",
        "variant": "Lineage B.1.1.519",
        "beta": "0.3500 - 0.5000",
        "gamma": "0.1200 - 0.1500",
        "delta": "0.0150 - 0.0250",
        "r0": "1.200 - 1.500",
    },
    {
        "phase": "Third Wave",
        "interval": "May 2021 - Nov 2021",
        "start": "2021-05-01", "end": "2021-11-30",
        "variant": "Gamma (P.1) / Delta (B.1.617.2)",
        "beta": "0.6000 - 0.8500",
        "gamma": "0.1400 - 0.1800",
        "delta": "0.0080 - 0.0150",
        "r0": "1.400 - 1.800",
    },
    {
        "phase": "Fourth Wave",
        "interval": "Dec 2021 - Mar 2022",
        "start": "2021-12-01", "end": "2022-03-31",
        "variant": "Omicron (BA.1 / BA.2)",
        "beta": "1.1000 - 1.6000",
        "gamma": "0.2000 - 0.3300",
        "delta": "0.0010 - 0.0030",
        "r0": "2.000 - 3.500",
    },
    {
        "phase": "Fifth Wave",
        "interval": "May 2022 - Oct 2022",
        "start": "2022-05-01", "end": "2022-10-31",
        "variant": "Omicron (BA.4 / BA.5)",
        "beta": "0.9000 - 1.3000",
        "gamma": "0.2200 - 0.3000",
        "delta": "0.0005 - 0.0015",
        "r0": "1.300 - 1.600",
    },
    {
        "phase": "Sixth Wave",
        "interval": "Nov 2022 - Mar 2023",
        "start": "2022-11-01", "end": "2023-03-31",
        "variant": "Omicron (BQ.1 / XBB)",
        "beta": "0.8000 - 1.2000",
        "gamma": "0.2500 - 0.3300",
        "delta": "0.0003 - 0.0010",
        "r0": "1.100 - 1.350",
    },
]

for wave in WAVES:
    wave.setdefault(
        "csv", f"filtered_data_{wave['start']}_{wave['end']}.csv"
    )

pd.DataFrame(WAVES)[["phase", "start", "end", "variant", "csv"]]

,phase,start,end,variant,csv
0,Pre-lockdown,2020-02-28,2020-03-22,Ancestral (Wuhan-Hu-1),pre_lockdown_filtered_data_2020-02-28_2020-03-...
1,First Wave,2020-03-23,2020-09-30,Ancestral lineages B.1,filtered_data_2020-03-23_2020-09-30.csv
2,Second Wave,2020-10-01,2021-04-30,Lineage B.1.1.519,filtered_data_2020-10-01_2021-04-30.csv
3,Third Wave,2021-05-01,2021-11-30,Gamma (P.1) / Delta (B.1.617.2),filtered_data_2021-05-01_2021-11-30.csv
4,Fourth Wave,2021-12-01,2022-03-31,Omicron (BA.1 / BA.2),filtered_data_2021-12-01_2022-03-31.csv
5,Fifth Wave,2022-05-01,2022-10-31,Omicron (BA.4 / BA.5),filtered_data_2022-05-01_2022-10-31.csv
6,Sixth Wave,2022-11-01,2023-03-31,Omicron (BQ.1 / XBB),filtered_data_2022-11-01_2023-03-31.csv


## Save one CSV per wave

In [4]:
summary = []

for wave in WAVES:
    block = filter_by_date(
        df, wave["start"], wave["end"],
        column_name="FECHA_INGRESO",
        positive=False, saved=False,
    )
    if POSITIVE_ONLY:
        block = covid_positive(block)

    path = WAVE_DIR / wave["csv"]
    if not wave.get("already_saved"):
        block.to_csv(path, index=False)

    summary.append({
        "Phase": wave["phase"],
        "Start": wave["start"],
        "End": wave["end"],
        "Records": len(block),
        "Positive cases": len(covid_positive(block)),
        "Deaths": int(block["FECHA_DEF"].notna().sum()),
        "CSV file": wave["csv"],
    })

summary = pd.DataFrame(summary)
summary

,Phase,Start,End,Records,Positive cases,Deaths,CSV file
0,Pre-lockdown,2020-02-28,2020-03-22,14528,1054,356,pre_lockdown_filtered_data_2020-02-28_2020-03-...
1,First Wave,2020-03-23,2020-09-30,1948639,813639,129841,filtered_data_2020-03-23_2020-09-30.csv
2,Second Wave,2020-10-01,2021-04-30,4782945,1564710,168800,filtered_data_2020-10-01_2021-04-30.csv
3,Third Wave,2021-05-01,2021-11-30,5325954,1558973,86435,filtered_data_2021-05-01_2021-11-30.csv
4,Fourth Wave,2021-12-01,2022-03-31,3683555,1821608,33489,filtered_data_2021-12-01_2022-03-31.csv
5,Fifth Wave,2022-05-01,2022-10-31,2715206,1356637,10591,filtered_data_2022-05-01_2022-10-31.csv
6,Sixth Wave,2022-11-01,2023-03-31,1157151,423432,7415,filtered_data_2022-11-01_2023-03-31.csv


In [5]:
parameters = pd.DataFrame([{
    "Epidemiological Phase": wave["phase"],
    "Time Interval": wave["interval"],
    "Start Date": wave["start"],
    "End Date": wave["end"],
    "Predominant Strain / Variant": wave["variant"],
    "Transmission Rate (β) [day^-1]": wave["beta"],
    "Recovery Rate (γ) [day^-1]": wave["gamma"],
    "Mortality Rate (δ or μ) [day^-1]": wave["delta"],
    "Reproductive Number (R0 / R(t))": wave["r0"],
    "CSV File": wave["csv"],
} for wave in WAVES])

METADATA_DIR.mkdir(parents=True, exist_ok=True)
parameters.to_csv(METADATA_DIR / "covid_waves_parameters.csv", index=False)
parameters

,Epidemiological Phase,Time Interval,Start Date,End Date,Predominant Strain / Variant,Transmission Rate (β) [day^-1],Recovery Rate (γ) [day^-1],Mortality Rate (δ or μ) [day^-1],Reproductive Number (R0 / R(t)),CSV File
0,Pre-lockdown,Feb 2020 - Mar 2020,2020-02-28,2020-03-22,Ancestral (Wuhan-Hu-1),0.5534 – 1.1400,0.0708 – 0.2000,0.0561 – 0.1200,1.762 – 2.860,pre_lockdown_filtered_data_2020-02-28_2020-03-...
1,First Wave,Mar 2020 - Sep 2020,2020-03-23,2020-09-30,Ancestral lineages B.1,0.5534 → 0.1069,0.0708 → 0.1383,0.0561 → 0.0211,1.100 - 1.300,filtered_data_2020-03-23_2020-09-30.csv
2,Second Wave,Sep 2020 - Apr 2021,2020-10-01,2021-04-30,Lineage B.1.1.519,0.3500 - 0.5000,0.1200 - 0.1500,0.0150 - 0.0250,1.200 - 1.500,filtered_data_2020-10-01_2021-04-30.csv
3,Third Wave,May 2021 - Nov 2021,2021-05-01,2021-11-30,Gamma (P.1) / Delta (B.1.617.2),0.6000 - 0.8500,0.1400 - 0.1800,0.0080 - 0.0150,1.400 - 1.800,filtered_data_2021-05-01_2021-11-30.csv
4,Fourth Wave,Dec 2021 - Mar 2022,2021-12-01,2022-03-31,Omicron (BA.1 / BA.2),1.1000 - 1.6000,0.2000 - 0.3300,0.0010 - 0.0030,2.000 - 3.500,filtered_data_2021-12-01_2022-03-31.csv
5,Fifth Wave,May 2022 - Oct 2022,2022-05-01,2022-10-31,Omicron (BA.4 / BA.5),0.9000 - 1.3000,0.2200 - 0.3000,0.0005 - 0.0015,1.300 - 1.600,filtered_data_2022-05-01_2022-10-31.csv
6,Sixth Wave,Nov 2022 - Mar 2023,2022-11-01,2023-03-31,Omicron (BQ.1 / XBB),0.8000 - 1.2000,0.2500 - 0.3300,0.0003 - 0.0010,1.100 - 1.350,filtered_data_2022-11-01_2023-03-31.csv


## Saved files

In [6]:
for wave in WAVES:
    path = WAVE_DIR / wave["csv"]
    size_mb = path.stat().st_size / 1e6 if path.exists() else float("nan")
    print(f"{wave['phase']:<14} {wave['start']} -> {wave['end']}  {size_mb:8.1f} MB  {path.name}")

assert (METADATA_DIR / "covid_waves_parameters.csv").exists()

Pre-lockdown   2020-02-28 -> 2020-03-22       1.3 MB  pre_lockdown_filtered_data_2020-02-28_2020-03-22.csv
First Wave     2020-03-23 -> 2020-09-30     178.4 MB  filtered_data_2020-03-23_2020-09-30.csv
Second Wave    2020-10-01 -> 2021-04-30     436.2 MB  filtered_data_2020-10-01_2021-04-30.csv
Third Wave     2021-05-01 -> 2021-11-30     485.4 MB  filtered_data_2021-05-01_2021-11-30.csv
Fourth Wave    2021-12-01 -> 2022-03-31     336.1 MB  filtered_data_2021-12-01_2022-03-31.csv
Fifth Wave     2022-05-01 -> 2022-10-31     249.7 MB  filtered_data_2022-05-01_2022-10-31.csv
Sixth Wave     2022-11-01 -> 2023-03-31     106.3 MB  filtered_data_2022-11-01_2023-03-31.csv
